In [ ]:
#!/usr/bin/env python3
"""
upload_results.py
- Reads S3 connection details from env vars (AWS_*)
- Creates a run_<YYYY-MM-DD_HH-MM> folder under TAXI_RUN_PREFIX
- Uploads processed/ plots/ metrics/ (and raw/manifest.json) back to S3
"""

# Install necessary libraries (no requirements.txt)
import sys
import subprocess
import importlib

def _ensure(import_name: str, pip_name: str) -> None:
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--user", pip_name])

_ensure("boto3", "boto3")
_ensure("botocore", "botocore")

# Import libraries
import os
from pathlib import Path
from datetime import datetime
import boto3
from botocore.client import Config

def _get_env(name: str) -> str:
    val = os.getenv(name, "").strip()
    if not val:
        raise RuntimeError(f"Missing required environment variable: {name}")
    return val

def _normalize_endpoint(endpoint: str) -> str:
    endpoint = endpoint.strip()
    if not endpoint.startswith("http://") and not endpoint.startswith("https://"):
        endpoint = "http://" + endpoint
    return endpoint

def _s3_client():
    key_id = _get_env("AWS_ACCESS_KEY_ID")
    secret_key = _get_env("AWS_SECRET_ACCESS_KEY")
    region = _get_env("AWS_DEFAULT_REGION")
    endpoint = _normalize_endpoint(_get_env("AWS_S3_ENDPOINT"))
    cfg = Config(signature_version="s3v4", s3={"addressing_style": "path"})
    return boto3.client(
        "s3",
        aws_access_key_id=key_id,
        aws_secret_access_key=secret_key,
        region_name=region,
        endpoint_url=endpoint,
        config=cfg,
    )

def _upload_dir(s3, bucket: str, s3_prefix: str, local_dir: Path) -> int:
    if not local_dir.exists():
        return 0
    count = 0
    for p in local_dir.rglob("*"):
        if p.is_file():
            rel = p.as_posix()
            key = f"{s3_prefix}/{rel}"
            s3.upload_file(str(p), bucket, key)
            count += 1
    return count

def main() -> None:
    s3 = _s3_client()
    bucket = _get_env("AWS_S3_BUCKET")

    run_prefix = os.getenv("TAXI_RUN_PREFIX", "results/nyc-taxi-etl").strip().rstrip("/")
    # Required format requested: run_<YYYY-MM-DD_HH-MM>
    stamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
    run_folder = f"run_{stamp}"

    # Optional: for uniqueness and traceability
    elyra_run = os.getenv("ELYRA_RUN_NAME", "").strip()
    if os.getenv("TAXI_APPEND_ELYRA_RUN", "false").lower() in ("1", "true", "yes") and elyra_run:
        run_folder = f"{run_folder}_{elyra_run}"

    s3_prefix = f"{run_prefix}/{run_folder}"

    # Upload only outputs you want to keep
    uploads = 0
    uploads += _upload_dir(s3, bucket, s3_prefix, Path("processed"))
    uploads += _upload_dir(s3, bucket, s3_prefix, Path("plots"))
    uploads += _upload_dir(s3, bucket, s3_prefix, Path("metrics"))

    # Include a small pointer to source input (not the whole raw CSV)
    manifest = Path("raw/manifest.json")
    if manifest.exists():
        s3.upload_file(str(manifest), bucket, f"{s3_prefix}/raw/manifest.json")
        uploads += 1

    print(f"Uploaded files: {uploads}", flush=True)
    print(f"Results location: s3://{bucket}/{s3_prefix}", flush=True)

if __name__ == "__main__":
    main()
